# Quantitative XAI Evaluation for AlphaDent YOLO26 (v2)

v2: robust to YOLO26's nested/e2e raw output format (fixes 'tuple indices' skip errors).
Order matters: run cells top-to-bottom — helpers define `unwrap_tensor`/`pred_class_and_score` used by the explainer.

1. **Deletion / insertion faithfulness** (RISE protocol) per CAM method.
2. **Inter-method agreement** vs prediction correctness (quantifies the paper's agreement claim).

Outputs: `xai_eval/` CSVs + figures + LaTeX rows, zipped.

In [ ]:
import os, glob, torch

# ---- Config ----
# Point at trained weights: local runs/, Kaggle working, or an attached weights dataset.
WEIGHTS_CANDIDATES = (
    glob.glob('runs/segment/*/train/weights/best.pt') +
    glob.glob('/kaggle/working/AlphaDentYolov26/runs/segment/*/train/weights/best.pt') +
    glob.glob('/kaggle/input/*/**/best.pt', recursive=True)
)
WEIGHTS_PATH = WEIGHTS_CANDIDATES[0] if WEIGHTS_CANDIDATES else None
assert WEIGHTS_PATH, 'No best.pt found — set WEIGHTS_PATH manually.'

DATA_DIR   = './AlphaDent'          # expects images/valid + labels/valid
IMAGE_SIZE = 640                     # match the resolution the chosen weights were trained at
N_IMAGES   = 30                      # eval subset size (raise if time allows)
N_STEPS    = 20                      # perturbation steps for deletion/insertion
OUT_DIR    = 'xai_eval'
DEVICE     = 0 if torch.cuda.is_available() else 'cpu'
METHODS    = ['EigenCAM', 'GradCAM', 'GradCAM++', 'XGradCAM', 'HiResCAM', 'LayerCAM', 'EigenGradCAM']

os.makedirs(OUT_DIR, exist_ok=True)
print(f'weights={WEIGHTS_PATH}\nimgsz={IMAGE_SIZE} n_images={N_IMAGES} device={DEVICE}')


In [ ]:
import os, cv2, numpy as np, torch

def letterbox(img, size, pad_val=114):
    h, w = img.shape[:2]
    r = size / max(h, w)
    nh, nw = int(h * r), int(w * r)
    rs = cv2.resize(img, (nw, nh))
    canvas = np.full((size, size, 3), pad_val, dtype=np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = rs
    return canvas, r, (top, left)

def to_tensor(img_bgr, device):
    x = img_bgr.astype(np.float32) / 255.0
    return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(device)

def unwrap_tensor(out):
    """Descend nested list/tuple output until first Tensor."""
    while isinstance(out, (list, tuple)):
        out = out[0]
    return out

def parse_yolo_preds(t, nc):
    """Normalize YOLO raw output. Returns (kind, mat):
       kind='raw': mat (C,N) channel-first, channels = [box4, cls nc, ...]
       kind='e2e': mat (N,6) decoded rows = [x1,y1,x2,y2,conf,cls]"""
    if t.ndim == 3:
        t = t[0]
    if t.ndim != 2:
        raise ValueError(f'unexpected pred shape {tuple(t.shape)}')
    r, c = t.shape
    if r >= 4 + nc and r < c:
        return 'raw', t
    if c >= 4 + nc and c < r:
        return 'raw', t.T
    if c == 6:
        return 'e2e', t
    if r == 6:
        return 'e2e', t.T
    raise ValueError(f'cannot parse pred shape {tuple(t.shape)} with nc={nc}')

def pred_class_and_score(t, nc, class_idx=None):
    """Differentiable: returns (class_idx, scalar score tensor for that class)."""
    kind, mat = parse_yolo_preds(t, nc)
    if kind == 'raw':
        if class_idx is None:
            class_idx = int(mat[4:4 + nc, :].sum(dim=-1).argmax())
        return class_idx, mat[4 + class_idx, :].max()
    det = mat
    if class_idx is None:
        class_idx = int(det[det[:, 4].argmax(), 5])
    m = det[:, 5].long() == class_idx
    score = det[m, 4].max() if m.any() else det[:, 4].sum() * 0.0
    return class_idx, score

@torch.no_grad()
def class_score(yolo, img_bgr, class_idx, device, nc):
    out = unwrap_tensor(yolo.model(to_tensor(img_bgr, device)))
    _, s = pred_class_and_score(out, nc, class_idx)
    return float(s.cpu())

def poly_txt_to_boxes(txt_path, img_w, img_h):
    boxes = []
    if not os.path.exists(txt_path):
        return boxes
    for line in open(txt_path):
        p = line.split()
        if len(p) < 7:
            continue
        cls = int(p[0])
        xs = np.array(p[1::2], dtype=float) * img_w
        ys = np.array(p[2::2], dtype=float) * img_h
        boxes.append((cls, xs.min(), ys.min(), xs.max(), ys.max()))
    return boxes

def box_iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

class YOLOExplainer:
    """
    Custom Class Activation Mapping (CAM) generator for Ultralytics YOLO models.
    Supports Eigen-CAM, Grad-CAM, Grad-CAM++, XGrad-CAM, HiRes-CAM, Layer-CAM, and EigenGrad-CAM.
    """
    def __init__(self, model, target_layer, method='eigencam'):
        self.model = model
        self.target_layer = target_layer
        self.method = method.lower()
        self.activations = None
        self.gradients = None
        
        # Register hooks
        self.forward_hook = target_layer.register_forward_hook(self._forward_hook_fn)
        if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
            self.backward_hook = target_layer.register_full_backward_hook(self._backward_hook_fn)
            
    def _forward_hook_fn(self, module, input, output):
        if isinstance(output, (list, tuple)):
            self.activations = output[0].clone()
        else:
            self.activations = output.clone()
            
    def _backward_hook_fn(self, module, grad_input, grad_output):
        if isinstance(grad_output, (list, tuple)):
            self.gradients = grad_output[0].clone()
        else:
            self.gradients = grad_output.clone()
            
    def remove_hooks(self):
        self.forward_hook.remove()
        if hasattr(self, 'backward_hook'):
            self.backward_hook.remove()
            
    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        
        with torch.set_grad_enabled(True):
            if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
                input_tensor.requires_grad = True
                
            # Ensure model weights are on the same device as input_tensor
            self.model.model.to(input_tensor.device)
            
            # Run raw PyTorch forward pass to track gradients and bypass Results wrapper
            outputs = self.model.model(input_tensor)
            
            preds = unwrap_tensor(outputs)
                
            if self.method == 'eigencam':
                act = self.activations.detach().cpu().numpy()[0]
                C, H, W = act.shape
                matrix = act.reshape(C, H * W).T
                matrix = matrix - np.mean(matrix, axis=0)
                U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
                projection = U[:, 0].reshape(H, W)
                heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
                return heatmap
                
            elif self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
                nc = getattr(self.model.model, 'nc', None) or len(self.model.names)
                class_idx, score = pred_class_and_score(preds, nc, class_idx)
                score.backward(retain_graph=True)
            
            act = self.activations.detach().cpu().numpy()[0]
            grad = self.gradients.detach().cpu().numpy()[0]
            
            if self.method == 'gradcam':
                weights = np.mean(grad, axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap
                
            elif self.method == 'gradcam++':
                grads_power_2 = grad ** 2
                grads_power_3 = grad ** 3
                sum_act = np.sum(act, axis=(1, 2))
                alpha = grads_power_2 / (2 * grads_power_2 + sum_act[:, None, None] * grads_power_3 + 1e-8)
                weights = np.sum(alpha * np.maximum(grad, 0), axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'xgradcam':
                sum_act = np.sum(act, axis=(1, 2))
                weights = np.sum(grad * act, axis=(1, 2)) / (sum_act + 1e-8)
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'hirescam':
                cam = np.sum(act * grad, axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'layercam':
                cam = np.sum(act * np.maximum(grad, 0), axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'eigengradcam':
                act_grad = act * np.maximum(grad, 0)
                C, H, W = act_grad.shape
                matrix = act_grad.reshape(C, H * W).T
                matrix = matrix - np.mean(matrix, axis=0)
                U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
                projection = U[:, 0].reshape(H, W)
                heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
                return heatmap

def show_heatmap(img_path, heatmap, title='Attention Map', save_path=None):
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, superimposed_img)
        
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].axis('off')
    axes[0].set_title('Original Image')
    
    axes[1].imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    axes[1].axis('off')
    axes[1].set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Deletion / Insertion faithfulness (RISE protocol) ----
import pandas as pd
from ultralytics import YOLO

model = YOLO(WEIGHTS_PATH)
model.model.to(DEVICE).eval()
nc = getattr(model.model, 'nc', None) or len(model.names)
target_layer = model.model.model[22]

val_images = sorted(glob.glob(f'{DATA_DIR}/images/valid/*.jpg') + glob.glob(f'{DATA_DIR}/images/valid/*.png'))[:N_IMAGES]
print(f'{len(val_images)} images, nc={nc}')

# Diagnostic: show raw output structure once
_probe = cv2.imread(val_images[0]); _lb, _, _ = letterbox(_probe, IMAGE_SIZE)
with torch.no_grad():
    _raw = model.model(to_tensor(_lb, DEVICE))
_t = unwrap_tensor(_raw)
print('raw output tensor shape:', tuple(_t.shape), '| parsed as:', parse_yolo_preds(_t, nc)[0])

def top_class(img_lb):
    with torch.no_grad():
        out = unwrap_tensor(model.model(to_tensor(img_lb, DEVICE)))
        ci, _ = pred_class_and_score(out, nc, None)
        return ci

def del_ins_auc(img_lb, heat, class_idx):
    h, w = img_lb.shape[:2]
    hm = cv2.resize(heat, (w, h)).flatten()
    order = np.argsort(-hm)
    base = class_score(model, img_lb, class_idx, DEVICE, nc) + 1e-8
    blur = cv2.GaussianBlur(img_lb, (51, 51), 0)
    flat_img, flat_blur = img_lb.reshape(-1, 3), blur.reshape(-1, 3)
    dele, ins = [1.0], [class_score(model, blur, class_idx, DEVICE, nc) / base]
    for k in range(1, N_STEPS + 1):
        idx = order[: int(len(order) * k / N_STEPS)]
        d = flat_img.copy(); d[idx] = 114
        i = flat_blur.copy(); i[idx] = flat_img[idx]
        dele.append(class_score(model, d.reshape(h, w, 3), class_idx, DEVICE, nc) / base)
        ins.append(class_score(model, i.reshape(h, w, 3), class_idx, DEVICE, nc) / base)
    return float(np.trapz(dele, dx=1 / N_STEPS)), float(np.trapz(ins, dx=1 / N_STEPS))

rows, heat_store = [], {}
for ip in val_images:
    img = cv2.imread(ip)
    img_lb, _, _ = letterbox(img, IMAGE_SIZE)
    try:
        cls = top_class(img_lb)
    except Exception as e:
        print('skip', ip, repr(e)); continue
    for m in METHODS:
        ex = YOLOExplainer(model, target_layer, method=m)
        try:
            heat = ex.generate(to_tensor(img_lb, DEVICE), class_idx=cls)
            ex.remove_hooks()
            d_auc, i_auc = del_ins_auc(img_lb, heat, cls)
            rows.append({'image': os.path.basename(ip), 'method': m, 'class': cls,
                         'deletion_auc': d_auc, 'insertion_auc': i_auc})
            heat_store[(ip, m)] = heat
        except Exception as e:
            ex.remove_hooks(); print('fail', m, ip, repr(e))
    if rows:
        pd.DataFrame(rows).to_csv(f'{OUT_DIR}/faithfulness_raw.csv', index=False)

assert rows, 'No results produced — check the diagnostic shape print above.'
df = pd.DataFrame(rows)
summary = df.groupby('method')[['deletion_auc', 'insertion_auc']].agg(['mean', 'std']).round(4)
summary.to_csv(f'{OUT_DIR}/faithfulness_summary.csv')
print(summary)


In [ ]:
# ---- Inter-method agreement + link to prediction correctness ----
# Agreement per image = mean pairwise Spearman correlation of the 7 heatmaps (64x64 grid).
# Correctness proxy = top prediction (conf 0.25) box-IoU>=0.5 with a same-class GT box.
from scipy.stats import spearmanr, mannwhitneyu
from itertools import combinations
import pandas as pd

def agreement(ip):
    hs = [cv2.resize(heat_store[(ip, m)], (64, 64)).flatten()
          for m in METHODS if (ip, m) in heat_store]
    if len(hs) < 2:
        return None
    cs = [spearmanr(a, b).correlation for a, b in combinations(hs, 2)]
    return float(np.nanmean(cs))

rows2 = []
for ip in val_images:
    ag = agreement(ip)
    if ag is None:
        continue
    img = cv2.imread(ip); H, W = img.shape[:2]
    r = model.predict(ip, imgsz=IMAGE_SIZE, conf=0.25, verbose=False, device=DEVICE)[0]
    correct, conf = 0, 0.0
    if len(r.boxes):
        b = r.boxes[r.boxes.conf.argmax()]
        conf = float(b.conf)
        pred = (int(b.cls), *b.xyxy[0].tolist())
        gts = poly_txt_to_boxes(f"{DATA_DIR}/labels/valid/{os.path.splitext(os.path.basename(ip))[0]}.txt", W, H)
        correct = int(any(g[0] == pred[0] and box_iou(pred[1:], g[1:]) >= 0.5 for g in gts))
    rows2.append({'image': os.path.basename(ip), 'agreement': ag, 'top_conf': conf, 'correct': correct})

ag_df = pd.DataFrame(rows2)
ag_df.to_csv(f'{OUT_DIR}/agreement.csv', index=False)
tp, fp = ag_df[ag_df.correct == 1].agreement, ag_df[ag_df.correct == 0].agreement
print(f"mean agreement: TP={tp.mean():.3f} (n={len(tp)})  FP/miss={fp.mean():.3f} (n={len(fp)})")
if len(tp) > 2 and len(fp) > 2:
    u, p = mannwhitneyu(tp, fp, alternative='greater')
    print(f"Mann-Whitney U (TP > FP): U={u:.0f}, p={p:.4f}")
print(f"Spearman(agreement, top_conf) = {spearmanr(ag_df.agreement, ag_df.top_conf).correlation:.3f}")


In [ ]:
# ---- Paper-ready outputs ----
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(f'{OUT_DIR}/faithfulness_raw.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.boxplot(column='deletion_auc', by='method', ax=axes[0], rot=45)
axes[0].set_title('Deletion AUC (lower = more faithful)'); axes[0].set_xlabel('')
df.boxplot(column='insertion_auc', by='method', ax=axes[1], rot=45)
axes[1].set_title('Insertion AUC (higher = more faithful)'); axes[1].set_xlabel('')
plt.suptitle(''); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_faithfulness.png', dpi=200); plt.show()

ag_df = pd.read_csv(f'{OUT_DIR}/agreement.csv')
plt.figure(figsize=(5, 4))
for c, lbl, col in [(1, 'correct (TP)', 'tab:green'), (0, 'incorrect/miss', 'tab:red')]:
    sub = ag_df[ag_df.correct == c]
    plt.scatter(sub.agreement, sub.top_conf, label=lbl, c=col, alpha=0.7)
plt.xlabel('inter-method agreement (mean pairwise Spearman)')
plt.ylabel('top prediction confidence'); plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_agreement.png', dpi=200); plt.show()

# LaTeX table body for the manuscript
s = pd.read_csv(f'{OUT_DIR}/faithfulness_summary.csv', header=[0, 1], index_col=0)
print('% --- paste into tab:faithfulness ---')
for m in s.index:
    print(f"{m} & {s.loc[m,('deletion_auc','mean')]:.3f} & {s.loc[m,('insertion_auc','mean')]:.3f} \\\\")

import shutil
shutil.make_archive('/kaggle/working/xai_eval_results' if os.path.exists('/kaggle/working') else 'xai_eval_results', 'zip', OUT_DIR)
print('zipped.')
